# MSG data

- Download required MSG data from a product list
- Add radiation data to an existing dataset

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os

import rasterio as rio
from pyproj import CRS

In [ ]:
from etdataset import msg
from etdataset.interpolation import create_grid_dataset
from etdataset.utils import read_product_list

## Inputs

In [ ]:
product_list = read_product_list(
    "landsat_products.csv", CRS(4326), "overlap_geometry"
)

In [ ]:
product_list

In [ ]:
output = "out"
os.makedirs(output, exist_ok=True)

## Download required data

In [ ]:
bounds = rio.coords.BoundingBox(*product_list.loc[0].geometry.bounds)
crs = CRS(4326)

In [ ]:
date = dt.datetime(2024, 10, 21, 10, 33)

Download daily DSSF

In [ ]:
msg.download_file(
    satellite=msg.MSGSatellite.MSG,
    product=msg.MSGProduct.DAILY_SURFACE_SOLAR_RADIATION_DOWNWARD,
    date=date,
    fmt=msg.MSGFormat.NETCDF,
    path=output,
)

In [ ]:
msg.download_file(
    satellite=msg.MSGSatellite.MSGIODC,
    product=msg.MSGProduct.DAILY_SURFACE_SOLAR_RADIATION_DOWNWARD,
    date=date,
    fmt=msg.MSGFormat.NETCDF,
    path=output,
)

Download DSSF

In [ ]:
msg.download_file(
    satellite=msg.MSGSatellite.MSG,
    product=msg.MSGProduct.SURFACE_SOLAR_RADIATION_DOWNWARD,
    date=date,
    fmt=msg.MSGFormat.NETCDF,
    path=output,
)

Download all files

In [ ]:
msg.download(date=date, latlon_bbox=bounds, path=output)

### Add data 

In [ ]:
# Create a test dataset
grid = create_grid_dataset(bounds, crs, 0.1)
# Add attributes
grid.attrs["vis_date"] = date.date()
grid.attrs["vis_time"] = date.time()
grid.attrs["tir_date"] = date.date()
grid.attrs["tir_time"] = date.time()
grid

In [ ]:
data = msg.add(grid, path=output)

In [ ]:
data

In [ ]:
data.rsd_msg.plot()

In [ ]:
data.daily_msg.plot()

In [ ]:
data.fdiff_msg.plot()

In [ ]:
data = msg.add(grid)